# Akan-BPE Phase 2A1 — Qwen3-0.6B × Akan TTS Tokenizer

Self-contained Colab notebook. Run all cells top-to-bottom.

**Steps:**
1. Clone repo and install deps
2. Download Akan datasets from HuggingFace Hub
3. Run QLoRA fine-tune experiment (`Qwen/Qwen3-0.6B` + Akan TTS tokenizer)
4. Inspect results — fertility reduction, eval loss/perplexity, generation samples

**GPU required.** Before running: Runtime → Change runtime type → T4 GPU.

In [1]:
# 1. Clone repo (skip if already cloned)
import os

REPO = "https://github.com/attabeezy/akan-bpe.git"
if not os.path.isdir("akan-bpe"):
    !git clone {REPO}
%cd akan-bpe

Cloning into 'akan-bpe'...
remote: Enumerating objects: 331, done.
remote: Counting objects: 100% (331/331), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 331 (delta 184), reused 257 (delta 115), pack-reused 0 (from 0)
Receiving objects: 100% (331/331), 529.35 KiB | 6.23 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/content/akan-bpe


In [2]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 91.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.1/513.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 14.4 MB/s eta 0:00:00
  Building editable for akan-bpe (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00:00:0100:01


In [3]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [4]:
# 4. Download Akan datasets from HuggingFace Hub
# Produces: data/pristine_twi_{train,validation,test}.jsonl
#           data/aka_asr_{train,validation,test}.jsonl
!python scripts/download.py --output-dir data

README.md: 29.2kB [00:00, 14.5MB/s]
Resolving data files: 100% 72/72 [00:00<00:00, 22346.45it/s]
Resolving data files: 100% 270/270 [00:00<00:00, 25251.12it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 13.1MB/s]
Wrote 150400 rows to data/pristine_twi_train.jsonl
Wrote 18800 rows to data/pristine_twi_validation.jsonl
Wrote 18800 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [5]:
# 5. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data" : Path("data/pristine_twi_train.jsonl"),
    "TTS test data"  : Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer"  : Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

All inputs ready.
  TTS train data: data/pristine_twi_train.jsonl  (240.1 MB)
  TTS test data: data/pristine_twi_test.jsonl  (30.1 MB)
  TTS tokenizer: models/tts_tokenizer.json  (0.5 MB)


In [6]:
# 6. Run Phase 2A1 experiment
# QLoRA fine-tune on Qwen3-0.6B with the Akan TTS tokenizer.
# Writes model adapters to models/phase2a1_qwen3_0_6b_tts/
# Writes result JSON to results/phase2a1_qwen3_0_6b_tts.json
!python scripts/model_integration.py \
    --experiment-id phase2a1_qwen3_0_6b_tts \
    --model-id Qwen/Qwen3-0.6B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a1_qwen3_0_6b_tts \
    --results-output results/phase2a1_qwen3_0_6b_tts.json \
    --device-mode colab-qlora \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

config.json: 100% 726/726 [00:00<00:00, 3.67MB/s]
tokenizer_config.json: 9.73kB [00:00, 25.5MB/s]
vocab.json: 2.78MB [00:00, 74.1MB/s]
merges.txt: 1.67MB [00:00, 98.1MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 11.1MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 1.50G/1.50G [00:12<00:00, 125MB/s] 
Loading weights: 100% 311/311 [00:01<00:00, 207.40it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100% 239/239 [00:00<00:00, 1.53MB/s]
Traceback (most recent call last):
  File "/content/akan-bpe/scripts/model_integration.py", line 112, in <module>
    main()
  File "/content/akan-bpe/scripts/model_integration.py", line 105, in main
    payloa

In [7]:
# 7. Load result JSON
import json
from pathlib import Path

result = json.loads(
    Path("results/phase2a1_qwen3_0_6b_tts.json").read_text(encoding="utf-8")
)
print("Experiment :", result["experiment_id"])
print("Model      :", result["model_id"])
print("Device     :", result.get("device", {}))

FileNotFoundError: [Errno 2] No such file or directory: 'results/phase2a1_qwen3_0_6b_tts.json'

In [ ]:
# 8. Token count comparison — fertility reduction
cmp = result["token_count_comparison"]
print(f"Base tokenizer fertility : {cmp['base_model_tokenizer']['fertility']:.3f} tokens/word")
print(f"Akan tokenizer fertility : {cmp['experiment_tokenizer']['fertility']:.3f} tokens/word")
print(f"Reduction ratio          : {cmp['token_reduction_ratio']:.1%}")

In [ ]:
# 9. Eval metrics
ev = result["eval"]
print(f"Eval loss  : {ev['eval_loss']:.4f}")
print(f"Perplexity : {ev['perplexity']:.2f}")

In [ ]:
# 10. Generation samples
for i, s in enumerate(result["generation_samples"], 1):
    print(f"--- Sample {i} ---")
    print(f"Prompt    : {s['prompt']}")
    print(f"Completion: {s['completion']}")
    print()